In [1]:
import os
print(os.getcwd())

d:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\caption_generation


## Testing using Conch_v1_5/TITAN

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
from huggingface_hub import login
from transformers import AutoModel 

login(token=os.environ["HF_TOKEN"])

titan = AutoModel.from_pretrained('MahmoodLab/TITAN', trust_remote_code=True)
conch, eval_transform = titan.return_conch()

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful


In [ ]:
import os
import torch
import h5py
import pandas as pd
from transformers import AutoModel

# === Inputs ===
patch_h5 = "your_patch_features.h5"  # H5 file with `features`, `coords`, and `patch_size_level0`
output_csv = "patch_zeroshot_predictions.csv"
labels = ["MSIH", "inflammation", "normal tissue", "fibrosis", "mitosis", "tumor"]
slide_name = "slide_XYZ"
top_k = 5

# === Load TITAN model ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True).to(device)

# === Prepare label prompts ===
label_dict = {label: i for i, label in enumerate(labels)}  # for sorting
sorted_labels = sorted(labels, key=lambda x: label_dict[x])
class_prompts = [x for x in sorted_labels]

with torch.autocast("cuda", torch.float16), torch.inference_mode():
    classifier = model.zero_shot_classifier(class_prompts, TEMPLATES, device=device)

# === Load patch features ===
with h5py.File(patch_h5, "r") as f:
    features = torch.tensor(f["features"][:], device=device)
    coords = torch.tensor(f["coords"][:])

# === Predict labels for each patch ===
results = []

with torch.autocast("cuda", torch.float16), torch.inference_mode():
    for i in range(features.shape[0]):
        feat = features[i].unsqueeze(0)  # single patch
        scores = model.zero_shot(feat, classifier).cpu().squeeze()
        top_indices = torch.topk(scores, k=top_k).indices
        top_scores = scores[top_indices]
        top_labels = [class_prompts[idx] for idx in top_indices]

        row = {
            "slide": slide_name,
            "patch_index": i,
            "coord_x": int(coords[i][0]),
            "coord_y": int(coords[i][1]),
        }
        for k in range(top_k):
            row[f"label{k+1}"] = top_labels[k]
            row[f"score{k+1}"] = round(top_scores[k].item(), 4)
        results.append(row)

# === Save to CSV ===
df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)
print(f"Saved predictions to {output_csv}")


In [16]:
# load example data
from huggingface_hub import hf_hub_download
import h5py
import os
import pickle
import yaml
from pathlib import Path
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel

TEMPLATES = [
    "CLASSNAME.",
    "a photomicrograph showing CLASSNAME.",
    "a photomicrograph of CLASSNAME.",
    "an image of CLASSNAME.",
    "an image showing CLASSNAME.",
    "an example of CLASSNAME.",
    "CLASSNAME is shown.",
    "this is CLASSNAME.",
    "there is CLASSNAME.",
    "a histopathological image showing CLASSNAME.",
    "a histopathological image of CLASSNAME.",
    "a histopathological photograph of CLASSNAME.",
    "a histopathological photograph showing CLASSNAME.",
    "shows CLASSNAME.",
    "presence of CLASSNAME.",
    "CLASSNAME is present.",
    "an H&E stained image of CLASSNAME.",
    "an H&E stained image showing CLASSNAME.",
    "an H&E image showing CLASSNAME.",
    "an H&E image of CLASSNAME.",
    "CLASSNAME, H&E stain.",
    "CLASSNAME, H&E."
]

device = torch.device("cpu")
model = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True).to(device)

# TEMPLATES = ["A photo of a CLASSNAME.", "This patch shows a CLASSNAME."]

# demo_h5_path = hf_hub_download(
#     "MahmoodLab/TITAN", 
#     filename="TCGA_demo_features/TCGA-PC-A5DK-01Z-00-DX1.C2D3BC09-411F-46CF-811B-FDBA7C2A295B.h5",
# )
# file = h5py.File(demo_h5_path, 'r')
# features = torch.from_numpy(file['features'][:])
# coords = torch.from_numpy(file['coords'][:])
# patch_size_lv0 = file['coords'].attrs['patch_size_level0']

In [17]:
# load configs and prompts for TCGA-OT task
with open('config_tcga-ot-V2.yaml', 'r') as file:
    task_config = yaml.load(file, Loader=yaml.FullLoader)
class_prompts = task_config['prompts']
target = task_config['target']
label_dict = task_config['label_dict']

In [18]:
# Test to check if correct yaml file
import yaml

# Load YAML file
with open('config_tcga-ot-V2.yaml', 'r') as file:
    task_config = yaml.load(file, Loader=yaml.FullLoader)

# Extract fields
class_prompts = task_config.get('prompts')
target = task_config.get('target')
label_dict = task_config.get('label_dict')

# Check if prompts loaded correctly
if not class_prompts or not isinstance(class_prompts, dict):
    raise ValueError("class_prompts not loaded properly. Check the YAML structure.")

# Print a sample for verification
print(f"Loaded {len(class_prompts)} class prompts:")
for i, (label, prompt) in enumerate(class_prompts.items()):
    print(f"  {i+1}. {label}: \"{prompt}\"")
    if i == 4:
        break  # Show only first 5 for brevity


Loaded 14 class prompts:
  1. Adipose-Fat: "adipose, adipose tissue, adipocytes, fat, fat cells"
  2. Debris-Necrosis: "debris, necrosis, dead cells"
  3. Lymphocytes: "inflammatory, lymphocytes, immune cells, infiltrating lymphocytes"
  4. Plasma Cells: "plasma cells"
  5. Lymphoid Aggregate: "lymphoid aggregate, lymphoid follicles"


In [ ]:
# # extract slide embedding
# with torch.autocast('cuda', torch.float16), torch.inference_mode():
#     features = features.to(device)
#     coords = coords.to(device)
#     slide_embedding = model.encode_slide_from_patch_features(features, coords, patch_size_lv0)

In [19]:
# create prompts for zero-shot classification
sorted_class_prompts = dict(sorted(class_prompts.items(), key=lambda item: label_dict.get(item[0], float('inf'))))
classes = list(sorted_class_prompts.keys())
class_prompts = [sorted_class_prompts[key] for key in sorted_class_prompts.keys()]
with torch.autocast('cuda', torch.float16), torch.inference_mode():
    classifier = titan.zero_shot_classifier(class_prompts, TEMPLATES, device=device)

In [20]:
# Save the classifier if need to
torch.save({
    'classifier': classifier,      # the text embeddings
    'class_prompts': class_prompts,
    'classes': classes,
    'templates': TEMPLATES
}, 'zero_shot_classifier_V2.pt')

In [ ]:
# Load a saved classifer if need to
checkpoint = torch.load('zero_shot_classifier.pt', map_location=device)
classifier = checkpoint['classifier']           # [num_classes, embed_dim]
class_prompts = checkpoint['class_prompts']
classes = checkpoint['classes']
templates = checkpoint['templates']

In [37]:
print(type(classifier))     # should be torch.Tensor
print(classifier.device)     # cuda:0 or cpu
classifier.shape  

<class 'torch.Tensor'>
cpu


torch.Size([768, 46])

In [63]:
import torch
from transformers import AutoModel
import pandas as pd

# === Load patch features ===
file_path = "TCGA-3L-AA1B_nonMSIH_patch_features.pt"
data = torch.load(file_path, map_location="cpu")

features = data['features'].squeeze(0)  # Now shape: [N, 768]
coords = data['coords'].squeeze(0)      # Now shape: [N, 2]

features = features[:15]
coords = coords[:15]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True).to(device)

# === Classify each patch feature individually ===
results = []
for i, feat in enumerate(features):
    feat = feat.to(device).unsqueeze(0)  # shape [1, D]
    with torch.inference_mode(), torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
        scores = model.zero_shot(feat, classifier)[0][0]  # shape [num_classes]
        top5 = torch.topk(scores, 5)
        for idx, (label_idx, score) in enumerate(zip(top5.indices, top5.values)):
            results.append({
                'patch_index': i,
                'x': coords[i][0].item(),
                'y': coords[i][1].item(),
                'label_rank': idx + 1,
                'label': class_prompts[label_idx],
                'score': score.item()
            })

# === Save to CSV ===
df = pd.DataFrame(results)
df.to_csv("patch_zero_shot_labels_15.csv", index=False)
print("Saved top-5 labels per patch to 'patch_zero_shot_labels_15.csv'")

Saved top-5 labels per patch to 'patch_zero_shot_labels_15.csv'


In [44]:
print(classifier.shape)        # Should be [768, num_classes]
print(classifier[:, 0][:5])    # First few values of first class embedding
print(classifier.norm(dim=1))  # Norms per class vector

torch.Size([768, 46])
tensor([-0.0362, -0.0068,  0.0036,  0.0322, -0.0339])
tensor([0.2460, 0.1376, 0.2295, 0.2186, 0.2266, 0.2960, 0.2670, 0.2430, 0.2670,
        0.3145, 0.2078, 0.2595, 0.2115, 0.1673, 0.2434, 0.3110, 0.2173, 0.2148,
        0.1783, 0.2499, 0.2461, 0.2120, 0.3336, 0.1869, 0.2390, 0.1495, 0.1979,
        0.1940, 0.1857, 0.2091, 0.1935, 0.3110, 0.2011, 0.2100, 0.1739, 0.1955,
        0.1888, 0.1656, 0.1766, 0.3250, 0.1899, 0.2232, 0.3660, 0.1516, 0.2139,
        0.1973, 0.1760, 0.1732, 0.2099, 0.3107, 0.2320, 0.2209, 0.2233, 0.2766,
        0.1718, 0.1860, 0.1869, 0.1844, 0.2575, 0.1846, 0.2325, 0.2037, 0.3295,
        0.3426, 0.3851, 0.2315, 0.2057, 0.3408, 0.2051, 0.1888, 0.2190, 0.3103,
        0.3306, 0.3807, 0.3124, 0.2137, 0.2041, 0.1768, 0.2225, 0.3604, 0.2932,
        0.2423, 0.2181, 0.2934, 0.2905, 0.1457, 0.2002, 0.2193, 0.2133, 0.1917,
        0.2312, 0.2723, 0.2651, 0.1813, 0.3343, 0.2837, 0.2164, 0.1845, 0.2178,
        0.2270, 0.1730, 0.2858, 0.2016, 0.37

In [ ]:
file_path = "TCGA-3L-AA1B_nonMSIH_patch_features.pt"
data = torch.load(file_path, map_location="cpu")

features = data['features'].squeeze(0)  # Now shape: [N, 768]

print("Feature shape:", features.shape)
print("First vector:", features[0][:5])
print("Norms:", features.norm(dim=1))
print("Min norm:", features.norm(dim=1).min())
print("Max norm:", features.norm(dim=1).max())

Feature shape: torch.Size([1878, 768])
First vector: tensor([ 0.1292,  1.8351, -2.6008, -1.3341, -0.5816])
Norms: tensor([27.7736, 27.7802, 27.7993,  ..., 27.7925, 27.8025, 27.8198])
Min norm: tensor(27.7490)
Max norm: tensor(27.8481)


In [ ]:
# # Normalize patch features
# features = features / features.norm(dim=1, keepdim=True)  # shape: [1878, 768]
# # Compute cosine similarity scores
# scores = features @ classifier  # shape: [1878, num_classes]

In [ ]:
# topk_scores, topk_indices = torch.topk(scores, 5, dim=1)  # shape: [1878, 5]

# # Get class labels from indices
# topk_labels = [[classes[idx] for idx in row] for row in topk_indices]

In [ ]:
# import pandas as pd

# slide_name = "TCGA-3L-AA1B"
# patch_ids = list(range(features.shape[0]))

# records = []
# for i in range(features.shape[0]):
#     for j in range(5):
#         records.append({
#             "slide_name": slide_name,
#             "patch_id": patch_ids[i],
#             "label": topk_labels[i][j],
#             "score": topk_scores[i][j].item()
#         })

# df = pd.DataFrame(records)
# df.to_csv(f"{slide_name}_patch_labels_top5.csv", index=False)
# print("Saved to CSV:", f"{slide_name}_patch_labels_top5.csv")


Saved to CSV: TCGA-3L-AA1B_patch_labels_top5.csv


In [ ]:
# import torch
# import pandas as pd

# # Assume these are already loaded:
# # features: [N, 768]
# # classifier: [768, C]  # class embeddings
# # classes: list of class names (length C)
# # slide_name: string (e.g., 'TCGA-3L-AA1B')

# # Step 1: Normalize features (if not already normalized)
# features = torch.nn.functional.normalize(features, dim=1)

# # Step 2: Get raw similarity scores
# scores = features @ classifier  # shape: [N, C]

# # Step 3: Apply softmax to each patch's scores to get probabilities
# probs = torch.softmax(scores, dim=1)  # shape: [N, C]

# # Step 4: Get top-k labels per patch
# topk = 5
# topk_probs, topk_indices = torch.topk(probs, k=topk, dim=1)  # each row: top 5 scores
# topk_labels = [[classes[i] for i in row] for row in topk_indices.tolist()]

# # Step 5: Prepare rows for CSV
# csv_rows = []
# for patch_idx, (labels, probs) in enumerate(zip(topk_labels, topk_probs.tolist())):
#     for label, prob in zip(labels, probs):
#         csv_rows.append({
#             'slide_name': slide_name,
#             'patch_num': patch_idx,
#             'label': label,
#             'confidence': round(prob, 4)
#         })

# # Step 6: Save to CSV
# df = pd.DataFrame(csv_rows)
# df.to_csv('patch_label_predictions.csv', index=False)
# print("Saved predictions to patch_label_predictions.csv")

Saved predictions to patch_label_predictions.csv


In [ ]:
# Slide Test 1
import torch

# Load the saved slide embedding
slide_embedding_path = r"D:\DataInsight\Label_matching\TCGA-3L-AA1B_nonMSIH_slide_embedding.pt"
slide_embedding = torch.load(slide_embedding_path, map_location=device)  # shape: [1, D]
slide_embedding = slide_embedding.to(device)

# Run zero-shot classification
with torch.inference_mode(), torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
    scores = model.zero_shot(slide_embedding, classifier)  # shape: [1, 1, num_classes]
    top5 = torch.topk(scores[0][0], 5)

# Print top-5 predictions
for rank, (label_idx, score) in enumerate(zip(top5.indices, top5.values), 1):
    print(f"{rank}. {class_prompts[label_idx]} — score: {score.item():.4f}")


1. ['colon adenocarcinoma', 'adenocarcinoma of the colon', 'colorectal adenocarcinoma', 'COAD'] — score: 0.0339
2. ['rectal adenocarcinoma', 'adenocarcinoma of the rectum', 'rectal cancer', 'READ'] — score: 0.0337
3. ['tubular stomach adenocarcinoma', 'tubular adenocarcinoma of the stomach', 'stomach tubular adenocarcinoma', 'gastric tubular adenocarcinoma', 'TSTAD'] — score: 0.0294
4. ['esophageal adenocarcinoma', 'adenocarcinoma of the esophagus', 'esophageal cancer, adenocarcinoma type', 'ESCA'] — score: 0.0291
5. ['diffuse type adenocarcinoma of the stomach', 'diffuse stomach adenocarcinoma', 'diffuse gastric adenocarcinoma', 'stomach adenocarcinoma, diffuse type', 'DSTAD'] — score: 0.0288


In [ ]:
# Slide Test 2
import torch

# Load the saved slide embedding
slide_embedding_path = r"D:\DataInsight\Label_matching\TCGA-A6-5665_MSIH_slide_embedding.pt"
slide_embedding = torch.load(slide_embedding_path, map_location=device)  # shape: [1, D]
slide_embedding = slide_embedding.to(device)

# Run zero-shot classification
with torch.inference_mode(), torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
    scores = model.zero_shot(slide_embedding, classifier)  # shape: [1, 1, num_classes]
    top5 = torch.topk(scores[0][0], 5)

# Print top-5 predictions
for rank, (label_idx, score) in enumerate(zip(top5.indices, top5.values), 1):
    print(f"{rank}. {class_prompts[label_idx]} — score: {score.item():.4f}")


1. ['mucinous adenocarcinoma', 'mucinous adenocarcinoma of the colon', 'mucinous adenocarcinoma of the rectum', 'mucinous colorectal adenocarcinoma', 'MACR'] — score: 0.0334
2. ['tubular stomach adenocarcinoma', 'tubular adenocarcinoma of the stomach', 'stomach tubular adenocarcinoma', 'gastric tubular adenocarcinoma', 'TSTAD'] — score: 0.0334
3. ['colon adenocarcinoma', 'adenocarcinoma of the colon', 'colorectal adenocarcinoma', 'COAD'] — score: 0.0332
4. ['esophageal adenocarcinoma', 'adenocarcinoma of the esophagus', 'esophageal cancer, adenocarcinoma type', 'ESCA'] — score: 0.0325
5. ['diffuse type adenocarcinoma of the stomach', 'diffuse stomach adenocarcinoma', 'diffuse gastric adenocarcinoma', 'stomach adenocarcinoma, diffuse type', 'DSTAD'] — score: 0.0320


In [59]:
# Slide Test 3
import torch

# Load the saved slide embedding
slide_embedding_path = r"D:\DataInsight\Label_matching\TCGA-A6-6653_MSIH_slide_embedding.pt"
slide_embedding = torch.load(slide_embedding_path, map_location=device)  # shape: [1, D]
slide_embedding = slide_embedding.to(device)

# Run zero-shot classification
with torch.inference_mode(), torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
    scores = model.zero_shot(slide_embedding, classifier)  # shape: [1, 1, num_classes]
    top5 = torch.topk(scores[0][0], 5)

# Print top-5 predictions
for rank, (label_idx, score) in enumerate(zip(top5.indices, top5.values), 1):
    print(f"{rank}. {class_prompts[label_idx]} — score: {score.item():.4f}")

1. ['mucinous adenocarcinoma', 'mucinous adenocarcinoma of the colon', 'mucinous adenocarcinoma of the rectum', 'mucinous colorectal adenocarcinoma', 'MACR'] — score: 0.0369
2. ['colon adenocarcinoma', 'adenocarcinoma of the colon', 'colorectal adenocarcinoma', 'COAD'] — score: 0.0356
3. ['rectal adenocarcinoma', 'adenocarcinoma of the rectum', 'rectal cancer', 'READ'] — score: 0.0334
4. ['tubular stomach adenocarcinoma', 'tubular adenocarcinoma of the stomach', 'stomach tubular adenocarcinoma', 'gastric tubular adenocarcinoma', 'TSTAD'] — score: 0.0284
5. ['diffuse type adenocarcinoma of the stomach', 'diffuse stomach adenocarcinoma', 'diffuse gastric adenocarcinoma', 'stomach adenocarcinoma, diffuse type', 'DSTAD'] — score: 0.0275


In [62]:
# Slide Test 4
import torch

# Load the saved slide embedding
slide_embedding_path = r"D:\DataInsight\Label_matching\TCGA-AA-3812_nonMSIH_slide_embedding.pt"
slide_embedding = torch.load(slide_embedding_path, map_location=device)  # shape: [1, D]
slide_embedding = slide_embedding.to(device)

# Run zero-shot classification
with torch.inference_mode(), torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
    scores = model.zero_shot(slide_embedding, classifier)  # shape: [1, 1, num_classes]
    top5 = torch.topk(scores[0][0], 5)

# Print top-5 predictions
for rank, (label_idx, score) in enumerate(zip(top5.indices, top5.values), 1):
    print(f"{rank}. {class_prompts[label_idx]} — score: {score.item():.4f}")

1. ['rectal adenocarcinoma', 'adenocarcinoma of the rectum', 'rectal cancer', 'READ'] — score: 0.0325
2. ['colon adenocarcinoma', 'adenocarcinoma of the colon', 'colorectal adenocarcinoma', 'COAD'] — score: 0.0317
3. ['tubular stomach adenocarcinoma', 'tubular adenocarcinoma of the stomach', 'stomach tubular adenocarcinoma', 'gastric tubular adenocarcinoma', 'TSTAD'] — score: 0.0288
4. ['diffuse type adenocarcinoma of the stomach', 'diffuse stomach adenocarcinoma', 'diffuse gastric adenocarcinoma', 'stomach adenocarcinoma, diffuse type', 'DSTAD'] — score: 0.0271
5. ['diffuse type stomach adenocarcinoma', 'diffuse gastric adenocarcinoma', 'diffuse-type gastric cancer', 'gastric adenocarcinoma, diffuse type', 'STAD'] — score: 0.0265


## Testing using Conch_v1

### zeroshot_classification_example_starter

In [15]:
import os
print(os.getcwd())

d:\DataInsight\Label_matching


In [16]:
from conch.open_clip_custom import create_model_from_pretrained, tokenize, get_tokenizer
import torch
import os
from PIL import Image
from pathlib import Path

# show all jupyter output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint_path = 'checkpoints/CONCH/pytorch_model.bin'
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device)
_ = model.eval()

In [ ]:
# Features already available so comment this out
# image = Image.open(r"D:\DataInsight\Label_matching\TCGA-3L-AA1B_nonMSIH_subset\TCGA-3L-AA1B_nonMSIH_x2608_y14048_patch00000.png")
# image_tensor = preprocess(image).unsqueeze(0).to(device)
# image.resize((224, 224))

In [28]:
tokenizer = get_tokenizer()
classes = ['invasive lobular carcinoma',
           'invasive ductal carcinoma']
prompts = ['an H&E image of invasive lobular carcinoma', 
           'an H&E image of invasive ductal carcinoma']
tokenized_prompts = tokenize(texts=prompts, tokenizer=tokenizer).to(device)
tokenized_prompts.shape

torch.Size([2, 128])

In [27]:
pt_path = r"D:\DataInsight\Label_matching\TCGA-3L-AA1B_nonMSIH_patch_features.pt"
pt_data = torch.load(pt_path)

features = pt_data['features']  # Shape: [1, 1800, 768]
features = features.squeeze(0)  # Now: [1800, 768]
patch0_feature = features[0].unsqueeze(0).to(device)  # Shape: [1, 768]
patch0_feature_512 = patch0_feature[:, :512]  # Shape: [1, 512]

with torch.inference_mode():
    text_embedings = model.encode_text(tokenized_prompts)
    sim_scores = (patch0_feature_512 @ text_embedings.T * model.logit_scale.exp()).softmax(dim=-1).cpu().numpy()

print("Predicted class:", classes[sim_scores.argmax()])
print("Normalized similarity scores:", [f"{cls}: {score:.3f}" for cls, score in zip(classes, sim_scores[0])])

Predicted class: invasive lobular carcinoma
Normalized similarity scores: ['invasive lobular carcinoma: 1.000', 'invasive ductal carcinoma: 0.000']


## PathCLIP Approach

In [73]:
import torch
from PIL import Image
import open_clip

# === Load model (PathCLIP checkpoint) ===
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-16',
    pretrained='checkpoints/PathClip/pathclip-base.pt',
    cache_dir='/mnt/Xsky/syx/model/open_clip',  # optional, if not relevant, you can remove
    force_quick_gelu=True
)

tokenizer = open_clip.get_tokenizer('ViT-B-16')
model = model.eval()  # remove .cuda()

# === Load image and text ===
img_path = 'TCGA-3L-AA1B_nonMSIH_x5680_y16096_patch00017.png'
label_description_list = ['Colon Adenocarcinoma', 'rectal adenocarcinoma']
text_label_list = [f'An image of {desc}' for desc in label_description_list]

image = Image.open(img_path).convert("RGB")
image = preprocess(image).unsqueeze(0)  # stay on CPU
text = tokenizer(text_label_list)       # stay on CPU

# === Inference (on CPU) ===
with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
    predict_label = torch.argmax(text_probs).item()

image_features.shape
text_features.shape

# === Print results ===
print(f"Predicted: {label_description_list[predict_label]}")
print("Scores:", text_probs.squeeze().tolist())

d:\miniconda_3\envs\titan_env\Lib\site-packages\torch\nn\modules\activation.py:1160: UserWarning: Converting mask without torch.bool dtype to bool; this will negatively affect performance. Prefer to use a boolean mask directly. (Triggered internally at ..\aten\src\ATen\native\transformers\attention.cpp:152.)
  return torch._native_multi_head_attention(


torch.Size([1, 512])

torch.Size([2, 512])

Predicted: rectal adenocarcinoma
Scores: [0.3982788920402527, 0.6017211079597473]


In [65]:
!pip install open_clip_torch

  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 335.7 kB/s eta 0:00:03
   ------------- -------------------------- 0.5/1.5 MB 335.7 kB/s eta 0:00:03
   -------------------- ------------------- 0.8/1.5 MB 419.4 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.5 MB 419.4 kB/s eta 0:00:02
   --------------------------- ------------ 1.0/1.5 MB 484.0 kB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.5 MB 484.0 kB/s eta 0:00:01
   ---------------------------------- ----- 1.3/1.5 MB 53